# Bronze — preserve the synthetic source

Origami Health · synthetic training only. Website: lab1. Planned block: 30 minutes.

Run selected prepared cells; do not provision services in class. Configure the full asset pack and runtime bindings before running. Live AIDP execution and classroom timing remain unverified.


## First-time user guide

**Learning objective:** Land seven synthetic source datasets while preserving their original values and recording where they came from.

**Before you begin:** Notebook 00 passed. The facilitator has enabled writes only in your assigned workshop output area.

**Key terms:** Bronze is the raw landing layer. Ingestion copies source data into the platform. Metadata describes the source and run. Delta is the table format used for these workshop outputs.

**Timebox:** This is the shared 30-minute block for notebooks 01 AND 02, not 30 minutes each. Read the explanations as you run the prepared cells. Optional exploration is not part of the core timebox.

**How to run:** There are 2 code cells below. Run them one at a time, from top to bottom, using the editor's run-cell control. Wait for completion, read the actual output and only then continue. Do not use Run all on your first pass. All runtime bindings are facilitator-prepared; do not paste credentials or edit another participant's paths.

For common problems, open START_HERE.md at the asset-pack root. If an assertion fails, stop and keep the error for the facilitator; do not change the assertion or expected values to make it pass.


## Load administrator-prepared participant bindings

**Purpose:** Load administrator-prepared participant bindings

**Inputs:** Full asset pack plus external ORIGAMI_ASSET_ROOT / ORIGAMI_RUNTIME_CONFIG bindings.

**Processing:** Run the provided bounded code in order.

**Validation:** Assertions must pass; save results and record any fallback.

**Expected output:** Assigned slot and synthetic snapshot; placeholders fail closed.

**Business meaning:** Governed synthetic data supports analysis, not coverage decisions.

**Your action — step 1:** Run the binding cell for this notebook, even if you already ran it in notebook 00; notebook sessions may be independent.

**Before moving on:** wait for completion, inspect the actual output against the expected result above, and stop if any error appears.


In [ ]:
import os, sys, json
from pathlib import Path
asset_root = os.environ.get('ORIGAMI_ASSET_ROOT')
if not asset_root:
    raise RuntimeError('Facilitator must configure ORIGAMI_ASSET_ROOT and ORIGAMI_RUNTIME_CONFIG before class.')
sys.path.insert(0, str(Path(asset_root) / 'shared'))
from origami_core import AS_OF, SOURCES, fingerprint, score_rows
from aidp_runtime import configuration, bounded_rows, persist, read_delta, source_frame, verify_fixture, unique, publish_snapshot, save_evidence
cfg = configuration()
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark.sparkContext.addPyFile(str(Path(asset_root) / 'shared' / 'origami_core.py'))
print('Participant:', cfg['participant_id'], '| synthetic snapshot:', AS_OF)

## Land the seven source datasets

**Purpose:** Land the seven source datasets

**Inputs:** Prepared CSV snapshots on the assigned asset path.

**Processing:** Run the provided bounded code in order.

**Validation:** Assertions must pass; save results and record any fallback.

**Expected output:** Seven Bronze snapshots with source/ingestion metadata; no corrections to source fields.

**Business meaning:** Governed synthetic data supports analysis, not coverage decisions.

**Your action — step 2:** Run the landing cell once. Locate source_record_id, claim_id, submitted_amount and _source_file in the displayed sample. Explain why the amount is still source text at this stage.

**Before moving on:** wait for completion, inspect the actual output against the expected result above, and stop if any error appears.


In [ ]:
from datetime import datetime, timezone
# Bind once on the driver: separate Spark actions must not regenerate this value.
ingested_at_utc = datetime.now(timezone.utc).isoformat()
source_keys={'members':['member_id'],'plans':['plan_id','plan_version'],'enrollments':['enrollment_id'],'providers':['provider_id'],'claims':['source_record_id'],'benefit_schedule':['benefit_id'],'benefit_usage':['usage_id']}
for name in SOURCES:
    raw=source_frame(spark,cfg,name)
    landed=(raw.withColumn('_source_file',F.lit(name+'.csv'))
        .withColumn('_ingested_at_utc',F.lit(ingested_at_utc))
        .withColumn('_run_id',F.lit('origami-v1-seed42'))
        .withColumn('_synthetic',F.lit('true')))
    result=persist(spark,cfg,'bronze_'+name,landed,source_keys[name])
    assert fingerprint(bounded_rows(result.select(*raw.columns)))==fingerprint(bounded_rows(raw))
read_delta(spark,cfg,'bronze_claims').select('source_record_id','claim_id','submitted_amount','_source_file').show(5,truncate=False)
save_evidence(cfg,'bronze',{'raw_claim_rows':read_delta(spark,cfg,'bronze_claims').count(),'source_datasets':len(SOURCES)})

## Completion checkpoint

- [ ] Every required code cell completed without error.
- [ ] I checked the actual result: Seven Bronze snapshots are written and reread. The sample contains original claim fields, and evidence reports 725 raw claim rows and 7 source datasets.
- [ ] I saved evidence: Record the raw count 725, source count 7 and one example source_record_id. The notebook also saves bronze.json under your assigned evidence folder.

**Discuss:** Why do we retain a bad source row instead of correcting it in Bronze?

**Self-check explanation:** Preserving the source makes the transformation auditable. Silver records explicit quality decisions and rejection reasons without silently rewriting source history.

**Next:** day1/02_silver.ipynb. Continue only after the core checkpoint passes, or after the facilitator explicitly records a labelled fallback.

## If you get stuck

Note the notebook name, cell heading and sanitized error. Missing configuration, permission errors and missing target tables are facilitator setup issues, not instructions to provision resources. Unexpected values are validation failures: do not erase tables, disable checks or rerun the entire pipeline blindly. See START_HERE.md for the troubleshooting table.
